# E1 Stage 1 cycle S1a — B4, simulator only, guard absent by omission (2026-09-15)

Code `/Users/terrancehamilton/reachy-1-2-sim-stage1` = `main` `6bca12e`. Legs: `setup_a` RAISE_TO_SIDE via `primitives.raise_to_side` (HOME->PRESENT), `flight_a` LOWER_TO_REST via `rig_motion.from_present` (PRESENT->REST). One attempt each; a `stop` marker or a failed start check means no motion.

In [1]:
# Cell 1 — connect with literals; hygiene shown
import os, sys, time, json, pathlib, traceback
sys.path.insert(0, "/Users/terrancehamilton/reachy-1-2-sim-stage1/src"); sys.path.insert(0, "/Users/terrancehamilton/reachy-1-2-sim-stage1/scripts")
CTRL = pathlib.Path("/Users/terrancehamilton/reachy-1-2-sim-stage1/docs/reviews/probes-2026-09-15-e1-stage1-b4/control"); RECORD_ROOT = "/Users/terrancehamilton/reachy-1-2-sim-stage1/docs/reviews/probes-2026-09-15-e1-stage1-b4/e1_server_runs"
SCENE = "/Users/terrancehamilton/reachy-1-2-sim-stage1/scenes/e1_boards/B4_pool_box_1_r2c3.yaml"; LEAD_IN_S = 3.0; CYCLE = "S1a"
print("REACHY env in this kernel:", {k: v for k, v in os.environ.items() if k.upper().startswith("REACHY")})
assert "REACHY_IP" not in os.environ and "REACHY_ENABLE_MOTION" not in os.environ, "shell hygiene violated"
from reachy_sdk import ReachySDK
HOST, PORT = "localhost", 50051
reachy = ReachySDK(host=HOST, sdk_port=PORT)
print(f"ReachySDK(host={HOST!r}, sdk_port={PORT}) connected at wall {time.time_ns()} mono {time.monotonic_ns()}")
print("python:", sys.executable)


REACHY env in this kernel: {}


ReachySDK(host='localhost', sdk_port=50051) connected at wall 1789506906449507000 mono 157249050860166
python: /private/tmp/claude-501/-Users-terrancehamilton-IITG-Reachy-Project/2ab0d1ac-fe72-429c-a2d9-192d80cab2b5/scratchpad/e1venv/bin/python


In [2]:
# Cell 2 — motion-client binding check: the recorder's identity function on THIS SDK object
import e1_identity
from reachy_ai.motion import rig_routes as R
from reachy_ai.motion import primitives
from reachy_ai.tasks import rig_motion
def _pose(): return {name: float(getattr(reachy.r_arm, name).present_position) for name in R.R_JOINTS}
ident = e1_identity.verify_simulator_identity(host=HOST, port=PORT, scene_path=SCENE, record_root=RECORD_ROOT, read_sdk_joints=_pose)
d = ident.as_dict(); print(json.dumps(d, indent=2, default=str))
BINDING_OK = bool(ident.ok)
(CTRL / (f"binding_ok_{CYCLE}" if BINDING_OK else f"binding_FAIL_{CYCLE}")).write_text(json.dumps(d, default=str))
print("BINDING_OK =", BINDING_OK); print("present:", {k: round(v, 1) for k, v in _pose().items()})
def wait_for(pred, timeout_s, period=0.25):
    t0 = time.monotonic()
    while time.monotonic() - t0 < timeout_s:
        if (CTRL / "stop").exists(): return "stop"
        if pred(): return "ready"
        time.sleep(period)
    return "timeout"
def start_check(kind):
    p = _pose(); here = R.posture_of(p)
    if kind == "PLACE_ROUTE_start": ok, why = rig_motion.check_start(reachy.r_arm, R.PLACE_ROUTE)
    elif kind == "PRESENT": ok = R.at_pose(p, R.PRESENT, tol=12.0, joints=list(R.GROSS_JOINTS)); why = "" if ok else "not at PRESENT (gross, 12 deg)"
    elif kind == "REST": ok = R.at_pose(p, R.REST, tol=12.0, joints=list(R.GROSS_JOINTS)); why = "" if ok else "not at REST (gross, 12 deg)"
    return {"kind": kind, "ok": bool(ok), "why": why, "posture_of": here, "pose": {k: round(v, 1) for k, v in p.items()}}
PREV_OK = BINDING_OK


{
  "ok": true,
  "reasons": [],
  "run_dir": "/Users/terrancehamilton/reachy-1-2-sim-stage1/docs/reviews/probes-2026-09-15-e1-stage1-b4/e1_server_runs/run_20260915_211004",
  "manifest": {
    "format_version": 1,
    "started_at": "2026-09-15T21:10:04.862240+00:00",
    "model_path": "/Users/terrancehamilton/reachy-1-2-sim-stage1/native_mujoco/model/reachy_1_2.xml",
    "model_sha256": "618ef2499f6207d5e9e72f8b9a4e538b0521008afc1a80f3ecfbab22ab35abb4",
    "scene_path": "/Users/terrancehamilton/reachy-1-2-sim-stage1/scenes/e1_boards/B4_pool_box_1_r2c3.yaml",
    "scene_sha256": "818d8e5755b4a8e892bf4b48a73a0321df717c3dcaf3c2313f9ed367056ab083",
    "scene_revision": "initial",
    "mujoco_version": "3.11.0",
    "python_version": "3.14.0 (v3.14.0:ebf955df7a8, Oct  7 2025, 08:20:14) [Clang 16.0.0 (clang-1600.0.26.6)]",
    "platform": "macOS-26.6.2-arm64-arm-64bit-Mach-O",
    "protocol_version": 1,
    "calibration_provenance": "measured_2026_08_27",
    "depth_enabled": false,
    "

In [3]:
# Leg setup_a: RAISE_TO_SIDE via primitives.raise_to_side (HOME->PRESENT) — one attempt, gated on go_setup_a + the recorder
LEG = {"leg": "setup_a", "route": "RAISE_TO_SIDE", "tool": "primitives.raise_to_side", "cycle": CYCLE}
go = wait_for(lambda: (CTRL / "go_setup_a").exists(), 1800) if PREV_OK else "not_eligible"
LEG["go"] = go; print("go:", go)
rec = wait_for(lambda: (CTRL / "recorder_setup_a.log").exists() and "fly the route now" in (CTRL / "recorder_setup_a.log").read_text(), 900) if go == "ready" else go
LEG["recorder_status"] = rec; print("recorder status:", rec)
if rec == "ready":
    time.sleep(LEAD_IN_S)
    LEG["start_check"] = start_check("PLACE_ROUTE_start"); print("start check:", LEG["start_check"])
if rec == "ready" and LEG["start_check"]["ok"]:
    LEG["t_start_mono_ns"] = time.monotonic_ns(); LEG["t_start_wall_ns"] = time.time_ns()
    phases = []
    reachy.turn_on("r_arm")
    try:
        ret = primitives.raise_to_side(reachy.r_arm)
        LEG["outcome"] = "returned"; LEG["returned"] = ret
    except Exception as exc:
        LEG["outcome"] = f"EXC {type(exc).__name__}: {exc}"; traceback.print_exc()
    LEG["phases"] = phases
    LEG["t_end_mono_ns"] = time.monotonic_ns(); LEG["t_end_wall_ns"] = time.time_ns()
    LEG["elapsed_s"] = (LEG["t_end_mono_ns"] - LEG["t_start_mono_ns"]) / 1e9
    LEG["end_pose"] = {k: round(v, 1) for k, v in _pose().items()}
    print("outcome:", LEG["outcome"], "elapsed %.1f s" % LEG["elapsed_s"]); print("returned:", LEG.get("returned")); print("end pose:", LEG["end_pose"])
else:
    LEG["outcome"] = "not_attempted"
PREV_OK = LEG["outcome"] == "returned"
(CTRL / "setup_a_done").write_text(json.dumps(LEG)); print(json.dumps(LEG, default=str))


go: ready


recorder status: ready


start check: {'kind': 'PLACE_ROUTE_start', 'ok': True, 'why': '', 'posture_of': 'home', 'pose': {'r_shoulder_pitch': -0.0, 'r_shoulder_roll': 0.0, 'r_arm_yaw': -0.0, 'r_elbow_pitch': 0.0, 'r_forearm_yaw': -0.0, 'r_wrist_pitch': -0.0, 'r_wrist_roll': 40.0, 'r_gripper': -39.3}}


outcome: EXC RuntimeError: the route out of the pocket stopped at BACK: r_elbow_pitch is 39 deg off, tolerance 6. The next waypoint's clearance was measured from this one, so it does not apply from here. elapsed 19.2 s
returned: None
end pose: {'r_shoulder_pitch': 39.3, 'r_shoulder_roll': -0.0, 'r_arm_yaw': 0.0, 'r_elbow_pitch': -39.0, 'r_forearm_yaw': 0.0, 'r_wrist_pitch': -0.1, 'r_wrist_roll': 40.1, 'r_gripper': -39.8}
{"leg": "setup_a", "route": "RAISE_TO_SIDE", "tool": "primitives.raise_to_side", "cycle": "S1a", "go": "ready", "recorder_status": "ready", "start_check": {"kind": "PLACE_ROUTE_start", "ok": true, "why": "", "posture_of": "home", "pose": {"r_shoulder_pitch": -0.0, "r_shoulder_roll": 0.0, "r_arm_yaw": -0.0, "r_elbow_pitch": 0.0, "r_forearm_yaw": -0.0, "r_wrist_pitch": -0.0, "r_wrist_roll": 40.0, "r_gripper": -39.3}}, "t_start_mono_ns": 157265538743500, "t_start_wall_ns": 1789506922937641000, "outcome": "EXC RuntimeError: the route out of the pocket stopped at BACK: r_el

Traceback (most recent call last):
  File "/var/folders/_3/10c3j8hd01z4pvclm0bxqxp00000gn/T/ipykernel_43905/1524438325.py", line 15, in <module>
    ret = primitives.raise_to_side(reachy.r_arm)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/terrancehamilton/reachy-1-2-sim-stage1/src/reachy_ai/motion/primitives.py", line 480, in raise_to_side
    fly(arm, R.PLACE_ROUTE, label="the route out of the pocket")
  File "/Users/terrancehamilton/reachy-1-2-sim-stage1/src/reachy_ai/motion/primitives.py", line 411, in fly
    raise RuntimeError(
RuntimeError: the route out of the pocket stopped at BACK: r_elbow_pitch is 39 deg off, tolerance 6. The next waypoint's clearance was measured from this one, so it does not apply from here.


In [4]:
# Leg flight_a: LOWER_TO_REST via rig_motion.from_present (PRESENT->REST) — one attempt, gated on go_flight_a + the recorder
LEG = {"leg": "flight_a", "route": "LOWER_TO_REST", "tool": "rig_motion.from_present", "cycle": CYCLE}
go = wait_for(lambda: (CTRL / "go_flight_a").exists(), 1800) if PREV_OK else "not_eligible"
LEG["go"] = go; print("go:", go)
rec = wait_for(lambda: (CTRL / "recorder_flight_a.log").exists() and "fly the route now" in (CTRL / "recorder_flight_a.log").read_text(), 900) if go == "ready" else go
LEG["recorder_status"] = rec; print("recorder status:", rec)
if rec == "ready":
    time.sleep(LEAD_IN_S)
    LEG["start_check"] = start_check("PRESENT"); print("start check:", LEG["start_check"])
if rec == "ready" and LEG["start_check"]["ok"]:
    LEG["t_start_mono_ns"] = time.monotonic_ns(); LEG["t_start_wall_ns"] = time.time_ns()
    phases = []
    reachy.turn_on("r_arm")
    try:
        ret = rig_motion.from_present(reachy.r_arm, on_phase=lambda *a: phases.append([time.monotonic_ns(), *map(str, a)]))
        LEG["outcome"] = "returned"; LEG["returned"] = ret
    except Exception as exc:
        LEG["outcome"] = f"EXC {type(exc).__name__}: {exc}"; traceback.print_exc()
    LEG["phases"] = phases
    LEG["t_end_mono_ns"] = time.monotonic_ns(); LEG["t_end_wall_ns"] = time.time_ns()
    LEG["elapsed_s"] = (LEG["t_end_mono_ns"] - LEG["t_start_mono_ns"]) / 1e9
    LEG["end_pose"] = {k: round(v, 1) for k, v in _pose().items()}
    print("outcome:", LEG["outcome"], "elapsed %.1f s" % LEG["elapsed_s"]); print("returned:", LEG.get("returned")); print("end pose:", LEG["end_pose"])
else:
    LEG["outcome"] = "not_attempted"
PREV_OK = LEG["outcome"] == "returned"
(CTRL / "flight_a_done").write_text(json.dumps(LEG)); print(json.dumps(LEG, default=str))


go: not_eligible
recorder status: not_eligible
{"leg": "flight_a", "route": "LOWER_TO_REST", "tool": "rig_motion.from_present", "cycle": "S1a", "go": "not_eligible", "recorder_status": "not_eligible", "outcome": "not_attempted"}


In [5]:
# Final read-only state; no further motion
print("final pose:", {k: round(v, 1) for k, v in _pose().items()}); print("done at wall", time.time_ns())


final pose: {'r_shoulder_pitch': 39.3, 'r_shoulder_roll': -0.0, 'r_arm_yaw': 0.0, 'r_elbow_pitch': -39.0, 'r_forearm_yaw': 0.0, 'r_wrist_pitch': -0.1, 'r_wrist_roll': 40.1, 'r_gripper': -39.8}
done at wall 1789506942170198000
